# 03 — Structured outputs

**Steer** — decide the *shape* of the answer, and the rules it must pass.

When you finish you can:

- get a validated Python object out of an agent instead of a string
- let the model choose between several output shapes
- add your own rule with an output validator and `ModelRetry`
- set the retry budget, and know what happens when it runs out

In [ ]:
from learningdantic import setup

MODEL = setup()

## A Pydantic model as the output

Describe the answer you want as a Pydantic model and pass it as `output_type`. Then `result.output` comes back
as that class, checked field by field. No parsing JSON out of a string.

Two Python notes: `class Hero(BaseModel):` defines a data shape, and each line like `name: str` is a field with
a **type hint** (the type its value must have). Pydantic checks those types for you.

(Under the hood, pydantic-ai gives the model a hidden "final result" tool and validates what it sends; the
[output docs](https://pydantic.dev/docs/ai/core-concepts/output/) cover the other modes.)

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent


class Hero(BaseModel):
    name: str
    power: str
    weakness: str


hero_agent = Agent(
    MODEL,
    output_type=Hero,
    instructions="Extract the hero profile from the user text. Invent nothing.",
)

result = await hero_agent.run(
    "Captain Cardboard — his power is folding into any shape; he loses strength near water."
)
hero = result.output  # a real Hero object, checked by Pydantic
print(hero)
print("it is a real object:", type(hero).__name__, "| name.upper() =", hero.name.upper())

> ✅ **Checkpoint** — you should see something like `name='Captain Cardboard' power='Folding into any shape'
> weakness='Loses strength near water'`, then `it is a real object: Hero | name.upper() = CAPTAIN CARDBOARD`.
> `result.output` is a `Hero`, not a string, so your editor can autocomplete `hero.name`.

## Let the model choose a shape

Pass a **list** of types and the model picks the one that fits each request. Check which one you got with
`isinstance`. (Notebook 11 uses this same trick to route work between agents.)

In [ ]:
class Fact(BaseModel):
    fact: str


class Question(BaseModel):
    question: str


sorter = Agent(
    MODEL,
    output_type=[Fact, Question],
    instructions="If the user states something, record it as a Fact; if they ask something, record it as a Question.",
)

r1 = await sorter.run("The Eiffel Tower is about 330 meters tall.")
r2 = await sorter.run("How tall is the Eiffel Tower?")

print("r1 →", type(r1.output).__name__, ":", r1.output)
print("r2 →", type(r2.output).__name__, ":", r2.output)
print("r1 is a Fact:", isinstance(r1.output, Fact))

> ✅ **Checkpoint** — `r1 → Fact : fact='…330 meters tall.'`, then
> `r2 → Question : question='How tall is the Eiffel Tower?'` and `r1 is a Fact: True`. The wording inside each
> can vary from run to run; the types don't. Same agent, two shapes.

## Your own rule: `output_validator` + `ModelRetry`

Pydantic checks the *shape* (types, required fields). For a rule about *meaning*, write an **output validator**:
a function that gets the output and either returns it or raises `ModelRetry("what to fix")`. `ModelRetry` sends
your message back to the model, and it tries again.

A line starting with `@` above a `def` is a **decorator**. `@caps_agent.output_validator` hands the function to
that agent, which calls it on every answer. The validator is a plain `def` that takes just the output, and
`-> Hero` after the parentheses is a type hint for what the function returns.

Each "try again" uses one retry. The default budget is **1** retry, so this agent raises it with `retries=3`.

In [ ]:
from pydantic_ai import ModelRetry

caps_agent = Agent(
    MODEL,
    output_type=Hero,
    instructions="Extract the hero profile from the user text. Invent nothing.",
    retries=3,  # allow up to 3 "try again" rounds (the default is 1)
)


@caps_agent.output_validator
def shout_the_name(output: Hero) -> Hero:
    """The hero registry wants names in ALL CAPS."""
    if not output.name.isupper():
        raise ModelRetry(
            f"The name must be ALL CAPS; '{output.name}' is not. Send it again in ALL CAPS."
        )
    return output


result = await caps_agent.run(
    "Captain Cardboard — his power is folding into any shape; he loses strength near water."
)
print(result.output)
print("model requests:", result.usage.requests)

> ✅ **Checkpoint** — you should see `name='CAPTAIN CARDBOARD' …` and usually `model requests: 2`. The model
> doesn't know your rule, so its first name wasn't ALL CAPS; your validator sent it back with your message, and
> the second answer passed. A higher count means it needed more tries, which `retries=3` allows.
>
> 🔭 **In Logfire** (optional) — this run's trace shows both model calls, so you can read the rejected first
> answer and the retry.

## When the retries run out

If the model still breaks your rule after the last retry, the run **raises `UnexpectedModelBehavior`** instead of
handing you bad data. To see it for free, attach the same validator to an agent on `TestModel` (the fake model
from 00). `TestModel` fills every text field with `'a'`, which is never ALL CAPS, so every answer is rejected.

In [ ]:
from pydantic_ai import UnexpectedModelBehavior
from pydantic_ai.models.test import TestModel

fake_agent = Agent(TestModel(), output_type=Hero)  # default budget: 1 retry
# Same as writing @fake_agent.output_validator above the def: a decorator is just a function call.
fake_agent.output_validator(shout_the_name)

try:
    await fake_agent.run(
        "Captain Cardboard — his power is folding into any shape; he loses strength near water."
    )
except UnexpectedModelBehavior as error:
    print("run failed:", error)

> ✅ **Checkpoint** — `run failed: Exceeded maximum output retries (1)`. One first answer, one retry, both
> rejected, then the run stops. In your own code, catch it where you can do something useful (show a message,
> log it, or try another way).

## 🏋️ Exercise — the recipe extractor

1. Define a `Recipe` model with `title: str`, `ingredients: list[str]`, `steps: list[str]` and `minutes: int`
   (`list[str]` means "a list of strings"). Build an agent that extracts a recipe from text you paste in.
2. Add an output validator that raises `ModelRetry` when `ingredients` is empty or `minutes <= 0`. Give the agent
   `retries=3`.

Stretch: also reject recipes with more than 10 ingredients, with a message that tells the model what to do instead.

Solution: [`../solutions/03_recipe_extractor.ipynb`](../solutions/03_recipe_extractor.ipynb)

## Next →

**[04 — Tools](04_tools.ipynb)**: give the agent your own functions to call.